In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# 显示设置
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


candidates = [
    Path("data/E Commerce Dataset.xlsx"),
    Path("./E Commerce Dataset.xlsx"),
    Path(r"C:\Users\Administrator\EDIT-1-24012456\data\E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请将文件放入data文件夹或修改路径")

# 读取工作表 E Comm
df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"读取文件路径：{DATA_PATH}")
print(f"数据集尺寸：{df.shape[0]} 行 , {df.shape[1]} 列")
print("\n数据前5行预览：")
display(df.head())

# 任务1 业务理解问题答案
"""
1. 单行数据含义：一条电商平台单个用户的会员、消费、APP使用、流失状态完整记录
2. 用户唯一标识字段：CustomerID
3. 用户流失目标字段：Churn（1=用户流失，0=用户留存）
"""

# ====================== 2. 缺失值统计报告 ======================
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({
    "缺失数量": missing_count,
    "缺失百分比(%)": missing_pct
}).sort_values("缺失数量", ascending=False)

print("\n===== 缺失值统计报告（按缺失量降序） =====")
display(missing_report)

# ====================== 3. 重复值检查 ======================
# 完全重复整行
total_dup_rows = df.duplicated().sum()
# 用户ID重复
customerid_dup = df["CustomerID"].duplicated().sum()

print(f"\n完全重复数据总行数：{total_dup_rows}")
print(f"CustomerID 用户ID重复记录数量：{customerid_dup}")

# 思考题解答
"""
不能直接删除CustomerID重复数据：
同一用户会产生多条不同时间周期的消费行为记录，重复ID代表用户多次下单，直接删除会丢失时序业务数据，影响流失分析。
"""

# ====================== 4. 数值字段中位数填充缺失值 ======================
num_fill_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

# 循环中位数填充
for col in num_fill_cols:
    df[col] = df[col].fillna(df[col].median())

print("\n中位数填充后，数值字段剩余缺失值：")
print(df[num_fill_cols].isna().sum())

# 思考题：中位数填充不适用场景
"""
1. 数据严重偏态，极值具备真实业务价值（大额返现、高频下单高价值用户），中位数会抹平用户差异；
2. 缺失并非随机，特定人群才产生缺失，填充中位数会扭曲原始数据分布；
替代方案：正态分布数据用均值填充、高相关字段用模型预测填充、分组中位数填充。
"""

# ====================== 5. 查看分类字段原始取值 ======================
cat_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]
print("\n===== 原始分类字段取值分布 =====")
for col in cat_cols:
    print(f"\n【{col}】")
    print(df[col].value_counts())

# ====================== 6. 统一同义分类文本 ======================
# 登录设备统一
df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace({
    "Phone": "Mobile Phone",
    "Mobile": "Mobile Phone"
})
# 支付方式统一
df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace({
    "COD": "Cash on Delivery",
    "CC": "Credit Card"
})

print("\n===== 标准化后分类字段分布 =====")
for col in cat_cols:
    print(f"\n【{col}】")
    print(df[col].value_counts())

# ====================== 7. IQR箱线法 统计异常值检测函数 ======================
def iqr_outlier_summary(data, col_name):
    series = data[col_name].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = ((series < lower_bound) | (series > upper_bound)).sum()
    res = pd.Series({
        "下四分位数Q1": q1,
        "上四分位数Q3": q3,
        "统计下限": lower_bound,
        "统计上限": upper_bound,
        "候选异常值数量": outlier_count
    }, name=col_name)
    return res

# 检测指定3个业务字段
print("\n===== IQR统计异常值检测结果 =====")
display(iqr_outlier_summary(df, "WarehouseToHome"))
display(iqr_outlier_summary(df, "OrderCount"))
display(iqr_outlier_summary(df, "CashbackAmount"))

"""
结论：IQR仅识别统计学候选异常，不能直接删除；部分极值是真实高价值用户行为，仅标记，需结合业务规则判断是否剔除。
"""

# ====================== 8. 业务逻辑异常数据校验 ======================
err_dict = {
    "使用时长小于0": (df["HourSpendOnApp"] < 0).sum(),
    "仓库距离小于0": (df["WarehouseToHome"] < 0).sum(),
    "订单数≤0": (df["OrderCount"] <= 0).sum(),
    "返现金额小于0": (df["CashbackAmount"] < 0).sum(),
}
print("\n===== 业务逻辑违规数据统计 =====")
display(pd.Series(err_dict))

# ====================== 9. 清洗验收断言校验 ======================
# 1. 数值字段无缺失
assert df[num_fill_cols].isna().sum().sum() == 0, "校验失败：数值字段仍存在缺失值！"
# 2. 登录设备旧简写已清理
assert "Phone" not in df["PreferredLoginDevice"].unique(), "校验失败：登录设备未完成标准化"
# 3. 支付方式旧简写已清理
assert "COD" not in df["PreferredPaymentMode"].unique(), "校验失败：支付方式未完成标准化"
assert "CC" not in df["PreferredPaymentMode"].unique(), "校验失败：支付方式未完成标准化"

print("\n✅ 全部数据清洗校验通过！")

# ====================== 10. 导出清洗完成数据集 ======================
# 自动创建output文件夹
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "ecommerce_customer_cleaned.csv"

# 导出csv，utf-8-sig防止Excel中文乱码
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"\n清洗完成！文件导出路径：{output_file.resolve()}")

读取文件路径：C:\Users\Administrator\EDIT-1-24012456\data\E Commerce Dataset.xlsx
数据集尺寸：5630 行 , 20 列

数据前5行预览：


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60



===== 缺失值统计报告（按缺失量降序） =====


,缺失数量,缺失百分比(%)
DaySinceLastOrder,307,5.45
OrderAmountHikeFromlastYear,265,4.71
Tenure,264,4.69
OrderCount,258,4.58
CouponUsed,256,4.55
HourSpendOnApp,255,4.53
WarehouseToHome,251,4.46
CustomerID,0,0.00
PreferredLoginDevice,0,0.00
Churn,0,0.00



完全重复数据总行数：0
CustomerID 用户ID重复记录数量：0

中位数填充后，数值字段剩余缺失值：
Tenure                         0
WarehouseToHome                0
HourSpendOnApp                 0
OrderAmountHikeFromlastYear    0
CouponUsed                     0
OrderCount                     0
DaySinceLastOrder              0
dtype: int64

===== 原始分类字段取值分布 =====

【PreferredLoginDevice】
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

【PreferredPaymentMode】
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

【PreferedOrderCat】
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64

===== 标准化后分类字段分布 =====

【PreferredLoginDevice】
PreferredLoginDevice
Mo

...

下四分位数Q1     1.00
上四分位数Q3     3.00
统计下限       -2.00
统计上限        6.00
候选异常值数量   703.00
Name: OrderCount, dtype: float64

下四分位数Q1   145.77
上四分位数Q3   196.39
统计下限       69.84
统计上限      272.33
候选异常值数量   438.00
Name: CashbackAmount, dtype: float64


===== 业务逻辑违规数据统计 =====


使用时长小于0    0
仓库距离小于0    0
订单数≤0      0
返现金额小于0    0
dtype: int64


✅ 全部数据清洗校验通过！

清洗完成！文件导出路径：C:\Users\Administrator\EDIT-1-24012456\day4_pandas1\output\ecommerce_customer_cleaned.csv
